# Portfolio Optimization with QAOA — a divi custom-workflow tutorial

This notebook walks through using **[divi](https://github.com/qoroquantum/divi)** to run a partitioned QAOA portfolio optimization. The pedagogical point: divi consumes any [`hybrid.traits.ProblemDecomposer`](https://docs.ocean.dwavesys.com/projects/hybrid/), so plugging in your own partitioning algorithm (here, Newman-2006 modularity-spectral) takes ~50 lines of glue and you get parallel execution + beam-search aggregation for free.

The notebook runs end-to-end locally on `MaestroSimulator`; one line swaps the backend for [QoroService](https://dash.qoroquantum.net) to run every partition concurrently on QPUs / cloud simulators.

## The problem

Portfolio optimization selects assets to **maximize return while minimizing risk**. Formulated as a QUBO:

$$\text{Minimize: } x^T \Sigma x - \lambda \mu^T x$$

where $\Sigma$ is the covariance matrix, $\mu$ is the returns vector, $x \in \{0,1\}^n$ are asset-selection bits, and $\lambda$ trades risk against return.

Small portfolios fit in a single QAOA. Real ones (484 S&P 500 assets) don't — they have to be partitioned into weakly-coupled clusters and each cluster solved as its own sub-QAOA. We use **modularity-spectral partitioning** on the asset-correlation graph to find those clusters.

In [ ]:
import time

import numpy as np

# Local default — runs every QAOA on MaestroSimulator.
from divi.backends import MaestroSimulator

# Cloud — comment the line above and uncomment below to run every partition in parallel
# on Qoro's cloud. Get an API key at https://dash.qoroquantum.net and put it in a
# `.env` file at the repo root as QORO_API_KEY first.
# from divi.backends import QoroService, JobConfig

---

## Phase 1 — Small Portfolio (Local)

Generate synthetic data for a small portfolio to prove the workflow.

In [ ]:
np.random.seed(42)
n_assets = 8

returns = np.random.uniform(0.01, 0.1, n_assets)
A = np.random.randn(n_assets, n_assets) * 0.1
covariance = A @ A.T + np.eye(n_assets) * 0.01  # positive definite

print(f"Portfolio: {n_assets} assets")
print(f"  returns range: [{returns.min():.4f}, {returns.max():.4f}]")
print(f"  covariance shape: {covariance.shape}")

In [ ]:
from utils import build_full_portfolio_qubo, evaluate_solution
import dimod

from divi.qprog import QAOA
from divi.qprog.problems import BinaryOptimizationProblem
from divi.qprog.optimizers import MonteCarloOptimizer

LAMBDA_PARAM = 0.75

qubo_matrix = build_full_portfolio_qubo(returns, covariance, lambda_param=LAMBDA_PARAM)
bqm = dimod.BinaryQuadraticModel(qubo_matrix, "BINARY")
print(f"QUBO: {len(bqm.variables)} variables")

**The divi primitives, briefly:**

- **`BinaryOptimizationProblem`** wraps a `dimod.BinaryQuadraticModel` as a divi problem. With no `decomposer` argument (Phase 1) it's solved as a single QAOA; with one (Phase 2) it gets partitioned.
- **`QAOA(...).run()`** is the single-program path. `n_layers=2` is the QAOA circuit depth (parameter $p$); `max_iterations=10` caps the classical outer loop.
- **`MonteCarloOptimizer`** is divi's sampling-based classical optimizer. `population_size=30` parameter sets per iteration, `n_best_sets=5` survivors carried forward.
- **`MaestroSimulator(shots=10_000)`** is divi's local backend; on the cloud, swap in `QoroService(...)`.

In [ ]:
backend = MaestroSimulator(shots=10_000)
# backend = QoroService(job_config=JobConfig(shots=10_000))  # ← cloud swap

optim = MonteCarloOptimizer(population_size=30, n_best_sets=5)

print("Running QAOA...")
t0 = time.time()

qaoa = QAOA(
    problem=BinaryOptimizationProblem(bqm),
    n_layers=2,
    optimizer=optim,
    max_iterations=10,
    backend=backend,
)
qaoa.run()

local_time = time.time() - t0
print(f"QAOA complete in {local_time:.1f}s")

In [ ]:
# Extract and evaluate solution
qaoa_solution = np.array(qaoa.solution, dtype=int)
print(f"QAOA solution: {qaoa_solution}")
print(f"Assets selected: {np.where(qaoa_solution == 1)[0]}")

evaluate_solution(qubo_matrix, qaoa_solution, returns, covariance)

---

## Phase 2 — Real Portfolio (S&P 500, 484 assets)

The 8-asset toy fits in a single QAOA. A real portfolio doesn't — it has to be **partitioned**. We use **modularity-spectral partitioning** on the asset correlation matrix to find weakly-coupled clusters, then solve each cluster as an independent QAOA via `PartitioningProgramEnsemble`. divi runs the partitions in parallel and stitches them back together with **beam-search aggregation**.

`ModularityDecomposer` (defined below) is a `hybrid.traits.ProblemDecomposer` adapter that plugs straight into `BinaryOptimizationProblem(decomposer=...)`. No bespoke solver loop, no manual aggregation.

In [ ]:
real_returns = np.load("2016-01-01_returns.npy")
real_covariance = np.load("2016-01-01_covariance.npy")
real_correlation = np.load("2016-01-01_correlation.npy")
print(f"Real portfolio: {len(real_returns)} assets (S&P 500, 2016-01-01)")

### How divi turns a custom partitioner into a workflow

The two cells below wire three things together — here's what each one is doing:

**`ModularityDecomposer(...)` — the custom seam.** divi accepts any [`hybrid.traits.ProblemDecomposer`](https://docs.ocean.dwavesys.com/projects/hybrid/) as the partitioning strategy, so *any* graph-partitioning algorithm becomes a custom workflow simply by wrapping it as one. The class defined below is ~30 lines of glue around `modularity_spectral_threshold` (Newman 2006, in `modularity_spectral_partitioning.py`): its `next()` method yields one sub-BQM at a time, and `BinaryOptimizationProblem` collects them via `hybrid.Unwind`. Drop in `hybrid.EnergyImpactDecomposer`, METIS, or your own algorithm — same plumbing.

**`hybrid.SplatComposer()` — the inverse.** Where the decomposer slices the BQM into sub-problems, the composer stitches per-partition solutions back into a global bitstring. `SplatComposer` is the trivial "drop each partition's bits into their global slots" version; for overlapping partitions you'd use a different composer.

**`PartitioningProgramEnsemble.aggregate_results(beam_width=k, n_partition_candidates=m)`.** Each partition's QAOA returns a distribution of bitstrings, not a single answer. `n_partition_candidates=m` keeps the top-`m` per partition; `beam_width=k` is the search width when combining them under the *full* QUBO energy. `beam_width=1` is greedy (concatenate each partition's best); larger widths trade compute for a better global optimum.

In [ ]:
import hybrid
from hybrid.exceptions import EndOfStream
from hybrid.utils import bqm_induced_by

from modularity_spectral_partitioning import modularity_spectral_threshold


class ModularityDecomposer(
    hybrid.traits.ProblemDecomposer, hybrid.traits.SISO, hybrid.Runnable
):
    """Yields modularity-spectral partitions of a BQM, one sub-BQM at a time.

    `hybrid.Unwind` repeatedly calls `next()` until `EndOfStream`, collecting
    every sub-problem; `BinaryOptimizationProblem` then exposes those as the
    ensemble's parallel sub-programs.

    Cache key is the BQM's variable set (cheap to compute, stable across
    `state.updated()` calls); BQM-value equality would iterate every quadratic
    bias on every `next()` and tank performance on dense problems.
    """

    def __init__(self, max_size=30, similarity_matrix=None, **runopts):
        super().__init__(**runopts)
        self.max_size = max_size
        if similarity_matrix is not None:
            similarity_matrix = np.asarray(similarity_matrix)
            if similarity_matrix.ndim != 2 or similarity_matrix.shape[0] != similarity_matrix.shape[1]:
                raise ValueError(f"similarity_matrix must be square 2D, got {similarity_matrix.shape}")
            if not np.all(np.isfinite(similarity_matrix)):
                raise ValueError("similarity_matrix contains non-finite values")
            if not np.allclose(similarity_matrix, similarity_matrix.T):
                raise ValueError("similarity_matrix must be symmetric")

        self.similarity_matrix = similarity_matrix
        self._cached_var_set = None
        self._iter_partitions = None

    def _compute_partitions(self, bqm):
        if self.similarity_matrix is not None:
            adj = np.abs(self.similarity_matrix)
            np.fill_diagonal(adj, 0.0)
        else:
            n = bqm.num_variables
            adj = np.zeros((n, n))
            for (u, v), j in bqm.quadratic.items():
                adj[u, v] = adj[v, u] = abs(j)

        variables = list(bqm.variables)
        if len(variables) != adj.shape[0]:
            raise ValueError(f"matrix shape {adj.shape} does not match BQM variable count {len(variables)}")

        communities, _ = modularity_spectral_threshold(adj, threshold=self.max_size)
        return [[variables[i] for i in c] for c in communities]

    def next(self, state, **runopts):
        bqm = state.problem
        var_set = frozenset(bqm.variables)
        if var_set != self._cached_var_set:
            self._cached_var_set = var_set
            self._iter_partitions = iter(self._compute_partitions(bqm))

        try:
            partition_vars = next(self._iter_partitions)
        except StopIteration:
            self._cached_var_set = None
            raise EndOfStream

        sample = state.samples.change_vartype(bqm.vartype).first.sample
        return state.updated(subproblem=bqm_induced_by(bqm, partition_vars, sample))

In [ ]:
from divi.qprog import EarlyStopping
from divi.qprog.workflows import PartitioningProgramEnsemble

MAX_PARTITION_SIZE = 20  # max assets per QAOA sub-problem

real_qubo = build_full_portfolio_qubo(
    real_returns, real_covariance, lambda_param=LAMBDA_PARAM
)

problem = BinaryOptimizationProblem(
    real_qubo,
    decomposer=ModularityDecomposer(
        max_size=MAX_PARTITION_SIZE,
        similarity_matrix=real_correlation,
    ),
    composer=hybrid.SplatComposer(),
)

ensemble = PartitioningProgramEnsemble(
    problem=problem,
    quantum_routine="qaoa",
    n_layers=2,
    optimizer=MonteCarloOptimizer(population_size=30, n_best_sets=5),
    max_iterations=10,
    early_stopping=EarlyStopping(patience=3),
    backend=backend,  # MaestroSimulator locally, QoroService on cloud — see imports cell
)

print("Decomposing portfolio + creating sub-programs...")
ensemble.create_programs()
print(f"  {len(ensemble.programs)} partitions created")

In [ ]:
print("Running every partition...")
t0 = time.time()
ensemble.run().join()
elapsed = time.time() - t0

n_parts = len(ensemble.programs)
print(f"Done in {elapsed:.1f}s.")
print(f"  partitions: {n_parts}")
print(f"  total circuits executed: {ensemble.total_circuit_count}")
print(f"  on a parallelizing backend (QoroService), all {n_parts} sub-QAOAs run")
print(f"  concurrently — sequentially this would scale ~{n_parts}x.")

solution, energy = ensemble.aggregate_results(beam_width=3, n_partition_candidates=5)
print(f"\nGlobal portfolio: {int(solution.sum())} assets selected, QUBO energy = {energy:.6f}")

### Scaling out to the cloud

Every partition above ran through `MaestroSimulator`. To run the same workflow on QPUs / cloud simulators with all partitions concurrent, flip the comments in the imports cell at the top:

```python
# from divi.backends import MaestroSimulator
from divi.backends import QoroService, JobConfig
```

…and uncomment the matching `backend = QoroService(...)` line in the Phase 1 setup cell. Nothing else changes.

Get an API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)**.

### Variation — same workflow, PCE engine instead of QAOA

[**PCE (Pauli Correlation Encoding)**](https://arxiv.org/abs/2401.09421) replaces QAOA's one-binary-variable-per-qubit mapping with a polynomial encoding that uses **logarithmically fewer qubits** for the same number of variables. A 20-asset partition compresses from 20 qubits to ⌈log₂(20+1)⌉ × 2 = 10. Useful when partition sizes still exceed your hardware's qubit count.

Everything in the workflow carries over — same `BinaryOptimizationProblem` (and therefore same `ModularityDecomposer` + `SplatComposer`), same `aggregate_results`. Only the engine kwargs change:

```python
import pennylane as qml
from divi.qprog import PCE, GenericLayerAnsatz
from divi.qprog.optimizers import PymooOptimizer, PymooMethod

ensemble = PartitioningProgramEnsemble(
    problem=problem,                         # ← unchanged: same decomposer/composer
    quantum_routine="pce",                   # ← was "qaoa"
    n_layers=3,
    optimizer=PymooOptimizer(PymooMethod.DE, population_size=30),
    max_iterations=15,
    early_stopping=EarlyStopping(patience=5),
    backend=backend,
    # PCE-specific engine kwargs:
    ansatz=GenericLayerAnsatz(
        gate_sequence=[qml.RY, qml.RZ],
        entangler=qml.CNOT,
        entangling_layout="all-to-all",
    ),
    encoding_type="poly",
)
ensemble.create_programs()
ensemble.run().join()
solution, energy = ensemble.aggregate_results(beam_width=3, n_partition_candidates=5)
```

This is the divi value prop: pick the engine that fits your hardware budget, leave the workflow alone.

---

## Ready for real-world portfolios?

484 assets, dozens of partitions, all optimized in parallel — **[get your API key](https://dash.qoroquantum.net)**.